# F2.3 — Validación técnica del dataset y verificación del código

**Proyecto Transversal MCDI500 — Grupo 8 — Universidad Andrés Bello**
Módulo: Programación para la Ciencia de Datos · Fase 2

---

## Propósito de este notebook

Verificar dos cosas distintas:

1. **El dataset resultante** cumple los contratos de calidad que las fases siguientes
   necesitan dar por garantizados.
2. **El código del pipeline** se comporta correctamente no sólo con los datos reales,
   sino también en **casos límite** y ante **entradas inválidas**.

## Estrategia de verificación

| Categoría | Qué comprueba | Datos usados |
|---|---|---|
| **Normal** | El resultado real cumple lo esperado | Dataset procesado en `F2_02` |
| **Límite** | Comportamiento en los bordes de cada regla | Datos sintéticos pequeños y controlados |
| **Excepción** | Las entradas inválidas fallan de forma controlada y explicable | Datos sintéticos defectuosos |

Los casos límite y de excepción usan **datos sintéticos** porque sólo así el resultado
esperado se conoce de antemano con exactitud. Todas las pruebas se registran en una
tabla final, que se exporta como evidencia de trazabilidad.

---
## 1. Configuración

In [1]:
import sys
from pathlib import Path


def raiz_proyecto() -> Path:
    """Localiza la raiz del repositorio a partir del directorio de trabajo."""
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "src" / "ponds").is_dir():
            return base
    raise RuntimeError(
        "No se encontro 'src/ponds'. Ejecute el notebook dentro del repositorio."
    )


RAIZ = raiz_proyecto()
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

import platform
from datetime import datetime

import numpy as np
import pandas as pd

from ponds import rutas, io_datos, limpieza, temporal
from ponds.validacion import ValidadorDataset, ErrorValidacion

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__}")
print("Ejecutado:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Python 3.10.11 | numpy 2.1.0 | pandas 2.2.2
Ejecutado: 2026-09-13 23:45:05


### 1.1 Herramientas de prueba

`probar()` ejecuta una prueba, captura su resultado y lo anota en el registro. Distingue
entre **FALLA** (una aserción no se cumplió: el código no hace lo esperado) y **ERROR**
(la prueba lanzó una excepción no prevista). `debe_lanzar()` verifica que una operación
inválida produzca exactamente la excepción esperada.

In [2]:
registro = []


def probar(categoria: str, nombre: str, funcion) -> None:
    """Ejecuta una prueba y anota su resultado en el registro."""
    try:
        detalle = funcion() or ""
        estado = "OK"
    except AssertionError as exc:
        estado, detalle = "FALLA", str(exc)
    except Exception as exc:  # noqa: BLE001
        estado, detalle = "ERROR", f"{type(exc).__name__}: {exc}"

    registro.append({"categoria": categoria, "prueba": nombre, "estado": estado, "detalle": detalle})
    print(f"[{estado:^5}] {categoria:<9} | {nombre}" + (f"\n          -> {detalle}" if detalle else ""))


def debe_lanzar(excepcion, funcion, contiene: str = None):
    """Verifica que `funcion` lance `excepcion` (y opcionalmente un mensaje)."""
    try:
        funcion()
    except excepcion as exc:
        if contiene is not None:
            assert contiene in str(exc), f"mensaje inesperado: {exc}"
        return exc
    raise AssertionError(f"se esperaba {excepcion.__name__} y no se lanzo")


def resultado_contrato(df: pd.DataFrame, codigo: str) -> str:
    """Devuelve 'OK' o 'FALLA' para un contrato concreto (p. ej. 'C07')."""
    informe = ValidadorDataset(df).validar()
    return informe.loc[informe["contrato"].str.startswith(codigo), "resultado"].iloc[0]


def crudo_sintetico(n: int = 72, estaciones=("station1", "Station2", "Station3")) -> pd.DataFrame:
    """Construye un CSV crudo sintetico con el mismo formato que Ponds data.csv."""
    ts = pd.date_range("2022-02-01", periods=n, freq="20min")
    partes = [
        pd.DataFrame(
            {
                "station": st,
                "Date": ts.strftime("%d-%m-%Y"),
                "Time": ts.strftime("%H:%M:%S"),
                "NITRATE(PPM)": "20.0", "PH": "7.2", "AMMONIA(mg/l)": "0.05", "TEMP": "26.5",
                "DO": "8.1", "TURBIDITY": "30.0", "MANGANESE(mg/l)": "0.6", "label": "0",
            }
        )
        for st in estaciones
    ]
    return pd.concat(partes, ignore_index=True)


def serie_con_hueco(inicio: int, largo: int, n: int = 12) -> pd.DataFrame:
    """Serie lineal regular de un estanque con un bloque de nulos controlado."""
    valores = np.linspace(20.0, 31.0, n)
    valores[inicio:inicio + largo] = np.nan
    return pd.DataFrame(
        {
            "timestamp": pd.date_range("2022-02-01", periods=n, freq="20min"),
            "estanque": "station1",
            "temperatura": valores,
        }
    )


print("Herramientas de prueba definidas.")

Herramientas de prueba definidas.


---
## 2. Carga del dataset procesado

In [3]:
df = io_datos.cargar_procesado()
df_escalado = io_datos.cargar_procesado("ponds_escalado.csv")
df_crudo = io_datos.cargar_crudo()

print(f"Procesado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Crudo:     {df_crudo.shape[0]:,} filas x {df_crudo.shape[1]} columnas")
df.dtypes.to_frame("dtype").T

Procesado: 74,854 filas x 22 columnas
Crudo:     74,796 filas x 11 columnas


,timestamp,estanque,nitrato,ph,amonio,temperatura,oxigeno_disuelto,turbidez,manganeso,anio,mes,dia,hora,minuto,dia_semana,nombre_dia,anio_mes,fecha,estacion_anio,franja_horaria,fila_imputada,etiqueta_calidad
dtype,datetime64[ns],category,float64,float64,float64,float64,float64,float64,float64,Int16,Int8,Int8,Int8,Int8,Int8,object,string[python],object,category,category,bool,Int8


---
## 3. Contratos de calidad sobre el dataset real

`ValidadorDataset` (`src/ponds/validacion.py`) evalúa diez contratos y **no se detiene
en el primer fallo**: ejecuta todos para que el diagnóstico sea completo en una sola
pasada.

In [4]:
validador = ValidadorDataset(df)
informe = validador.validar()
informe

,contrato,resultado,detalle
0,C01 - Dataset no vacio,OK,74854 filas
1,C02 - Esquema completo,OK,todas presentes
2,C03 - Tipos de dato correctos,OK,correctos
3,C04 - Timestamp sin nulos,OK,0 nulos
4,"C05 - Clave (estanque, timestamp) unica",OK,0 duplicados
5,C06 - Orden temporal monotono,OK,todas las series ordenadas
6,C07 - Valores dentro del rango fisico,OK,todos dentro de rango
7,C08 - Estanques presentes,OK,"3: ['station1', 'station2', 'station3']"
8,C09 - Nulos residuales controlados,OK,0.195% promedio (umbral 5.0%)
9,C10 - Cobertura temporal esperada,OK,2022-02-01 a 2023-01-21


In [5]:
# Puerta de salida: lanza ErrorValidacion si cualquier contrato falla.
validador.assert_valido()
print(f"El dataset procesado supera los {len(informe)} contratos de calidad.")

El dataset procesado supera los 10 contratos de calidad.


---
## 4. Casos normales

Comprobaciones de coherencia entre el dataset crudo y el procesado, que van más allá de
los contratos: verifican que el pipeline hizo **exactamente** lo que declara.

In [6]:
def n01():
    assert validador.aprobado(), validador.fallidos().to_string()
    return f"{len(informe)} contratos OK"


def n02():
    # Las filas reales del procesado deben ser las del crudo menos las descartadas
    # de forma documentada: filas vacias y filas con fecha pero sin hora.
    vacias = int(df_crudo.isna().all(axis=1).sum())
    sin_hora = int((df_crudo["Time"].isna() & ~df_crudo.isna().all(axis=1)).sum())
    esperado = len(df_crudo) - vacias - sin_hora
    reales = int((~df["fila_imputada"]).sum())
    assert reales == esperado, f"{reales} filas reales, se esperaban {esperado}"
    return f"{reales:,} = {len(df_crudo):,} - {vacias} vacias - {sin_hora} sin hora"


def n03():
    for est, g in df.groupby("estanque", observed=True):
        pasos = g["timestamp"].diff().dropna().unique()
        assert len(pasos) == 1 and pasos[0] == pd.Timedelta("20min"), f"{est}: pasos {pasos}"
    return "paso constante de 20 min en los 3 estanques"


def n04():
    t_min = rutas.RANGOS_FISICOS["temperatura"][0]
    assert df["temperatura"].min() >= t_min, f"minimo {df['temperatura'].min()}"
    assert all(pd.api.types.is_float_dtype(df[c]) for c in rutas.VARIABLES)
    return f"temperatura minima {df['temperatura'].min():.2f} C; variables float64"


def n05():
    # Todo nulo residual debe pertenecer a un hueco mas largo que el limite interpolable.
    for var in rutas.VARIABLES:
        rachas = df.groupby("estanque", observed=True)[var].transform(temporal.largo_rachas_nulas)
        residuales = rachas[df[var].isna()]
        if len(residuales):
            assert residuales.min() > rutas.MAX_HUECOS_INTERPOLABLES, (
                f"{var}: nulo en hueco de {residuales.min()} lecturas"
            )
    return f"todos los nulos residuales estan en huecos > {rutas.MAX_HUECOS_INTERPOLABLES} lecturas"


def n06():
    stats = df_escalado.groupby("estanque", observed=True)[rutas.VARIABLES].agg(["mean", "std"])
    medias = stats.xs("mean", axis=1, level=1).abs().max().max()
    desvios = (stats.xs("std", axis=1, level=1) - 1).abs().max().max()
    assert medias < 1e-6 and desvios < 1e-6, f"media max {medias}, |std-1| max {desvios}"
    return "media 0 y desviacion 1 por estanque"


def n07():
    resultado = limpieza.PipelineLimpieza(crudo_sintetico(), verboso=False).ejecutar()
    v = ValidadorDataset(resultado)
    assert v.aprobado(), v.fallidos().to_string()
    assert len(resultado) == 72 * 3
    return "dataset sintetico de 3 estanques supera los contratos"


probar("Normal", "N01 Dataset procesado supera todos los contratos", n01)
probar("Normal", "N02 Conservacion de registros crudo -> procesado", n02)
probar("Normal", "N03 Rejilla temporal regular por estanque", n03)
probar("Normal", "N04 Sin valores imposibles y tipos correctos", n04)
probar("Normal", "N05 Nulos residuales solo en huecos largos", n05)
probar("Normal", "N06 Escalado z-score correcto por estanque", n06)
probar("Normal", "N07 Pipeline completo sobre datos sinteticos validos", n07)

[ OK  ] Normal    | N01 Dataset procesado supera todos los contratos
          -> 10 contratos OK
[ OK  ] Normal    | N02 Conservacion de registros crudo -> procesado
          -> 74,707 = 74,796 - 38 vacias - 51 sin hora
[ OK  ] Normal    | N03 Rejilla temporal regular por estanque
          -> paso constante de 20 min en los 3 estanques
[ OK  ] Normal    | N04 Sin valores imposibles y tipos correctos
          -> temperatura minima 14.50 C; variables float64


[ OK  ] Normal    | N05 Nulos residuales solo en huecos largos
          -> todos los nulos residuales estan en huecos > 3 lecturas
[ OK  ] Normal    | N06 Escalado z-score correcto por estanque
          -> media 0 y desviacion 1 por estanque


[ OK  ] Normal    | N07 Pipeline completo sobre datos sinteticos validos
          -> dataset sintetico de 3 estanques supera los contratos


---
## 5. Casos límite

Cada regla del pipeline tiene un umbral. Estas pruebas comprueban el comportamiento
**exactamente en el borde**, que es donde se concentran los errores de tipo
"menor que" frente a "menor o igual que".

In [7]:
LIMITE = rutas.MAX_HUECOS_INTERPOLABLES   # 3 lecturas = 1 hora


def l01():
    base = serie_con_hueco(inicio=4, largo=LIMITE)
    resultado = temporal.interpolar_por_grupo(base, columnas=["temperatura"])
    esperado = np.linspace(20.0, 31.0, 12)
    assert resultado["temperatura"].notna().all(), "quedaron nulos en un hueco interpolable"
    assert np.allclose(resultado["temperatura"], esperado), "la interpolacion no es lineal"
    return f"hueco de {LIMITE} lecturas: rellenado y exacto"


def l02():
    base = serie_con_hueco(inicio=4, largo=LIMITE + 1)
    resultado = temporal.interpolar_por_grupo(base, columnas=["temperatura"])
    n_nulos = int(resultado["temperatura"].isna().sum())
    assert n_nulos == LIMITE + 1, f"se rellenaron {LIMITE + 1 - n_nulos} valores de un hueco largo"
    return f"hueco de {LIMITE + 1} lecturas: intacto ({n_nulos} nulos)"


def l03():
    base = serie_con_hueco(inicio=0, largo=2)
    base.loc[11, "temperatura"] = np.nan
    resultado = temporal.interpolar_por_grupo(base, columnas=["temperatura"])
    assert resultado["temperatura"].iloc[[0, 1, 11]].isna().all(), "se extrapolo en un extremo"
    return "no extrapola antes de la primera ni despues de la ultima lectura"


def l04():
    lo, hi = rutas.RANGOS_FISICOS["temperatura"]
    valores = [lo, hi, lo - 0.01, hi + 0.01]
    base = pd.DataFrame({"temperatura": valores})
    resultado = limpieza.PipelineLimpieza(base, verboso=False).anular_fallos_de_sensor().df
    t = resultado["temperatura"]
    assert t.iloc[0] == lo and t.iloc[1] == hi, "se anulo un valor en el borde permitido"
    assert t.iloc[2:].isna().all(), "no se anulo un valor fuera del rango"
    return f"[{lo}, {hi}] se conservan; {lo - 0.01} y {hi + 0.01} se anulan"


def l05():
    crudo = crudo_sintetico(n=1, estaciones=("station1",))
    resultado = limpieza.PipelineLimpieza(crudo, verboso=False).ejecutar()
    assert len(resultado) == 1, f"{len(resultado)} filas"
    fallidos = ValidadorDataset(resultado).validar().query("resultado == 'FALLA'")["contrato"].str[:3].tolist()
    assert fallidos == ["C08"], f"contratos fallidos: {fallidos}"
    return "1 registro atraviesa el pipeline; solo falla C08 (se esperan 3 estanques)"


def l06():
    base = pd.DataFrame({"estanque": ["station1"] * 5, "temperatura": [25.0] * 5})
    for metodo in limpieza.METODOS_ESCALADO:
        resultado = limpieza.escalar_variables(base, columnas=["temperatura"], metodo=metodo)
        assert resultado["temperatura"].isna().all(), f"{metodo}: resultado {resultado['temperatura'].tolist()}"
        assert not np.isinf(resultado["temperatura"]).any()
    return "serie constante: resultado NaN, nunca infinito"


probar("Limite", "L01 Hueco de exactamente 1 h se interpola", l01)
probar("Limite", "L02 Hueco de 1 h + 20 min no se interpola", l02)
probar("Limite", "L03 Nulos en los extremos no se extrapolan", l03)
probar("Limite", "L04 Valores en el borde exacto del rango fisico", l04)
probar("Limite", "L05 Dataset de un unico registro", l05)
probar("Limite", "L06 Escalado de una serie constante", l06)

[ OK  ] Limite    | L01 Hueco de exactamente 1 h se interpola
          -> hueco de 3 lecturas: rellenado y exacto
[ OK  ] Limite    | L02 Hueco de 1 h + 20 min no se interpola
          -> hueco de 4 lecturas: intacto (4 nulos)
[ OK  ] Limite    | L03 Nulos en los extremos no se extrapolan
          -> no extrapola antes de la primera ni despues de la ultima lectura
[ OK  ] Limite    | L04 Valores en el borde exacto del rango fisico
          -> [5.0, 50.0] se conservan; 4.99 y 50.01 se anulan
[ OK  ] Limite    | L05 Dataset de un unico registro
          -> 1 registro atraviesa el pipeline; solo falla C08 (se esperan 3 estanques)
[ OK  ] Limite    | L06 Escalado de una serie constante
          -> serie constante: resultado NaN, nunca infinito


---
## 6. Casos de excepción

Entradas inválidas o datos corruptos. Lo que se verifica no es que "no falle", sino que
**falle de la forma correcta**: con la excepción adecuada y un mensaje que explique el
problema, o marcando el contrato de calidad que corresponde.

In [8]:
def x01():
    exc = debe_lanzar(FileNotFoundError, lambda: io_datos.cargar_crudo("no_existe.csv"), contiene="Descargue")
    return f"FileNotFoundError con instrucciones de descarga"


def x02():
    vacio = df.iloc[0:0]
    assert resultado_contrato(vacio, "C01") == "FALLA"
    exc = debe_lanzar(ErrorValidacion, lambda: ValidadorDataset(vacio).assert_valido())
    return f"C01 FALLA y assert_valido lanza ErrorValidacion"


def x03():
    sin_columna = df.drop(columns=["oxigeno_disuelto"])
    assert resultado_contrato(sin_columna, "C02") == "FALLA"
    return "columna ausente detectada por C02"


def x04():
    corrupto = df.copy()
    corrupto["ph"] = corrupto["ph"].astype(str)
    assert resultado_contrato(corrupto, "C03") == "FALLA"
    return "variable como texto detectada por C03"


def x05():
    duplicado = pd.concat([df, df.iloc[[0]]], ignore_index=True)
    assert resultado_contrato(duplicado, "C05") == "FALLA"
    return "clave (estanque, timestamp) repetida detectada por C05"


def x06():
    fuera = df.copy()
    fuera.loc[fuera.index[0], "temperatura"] = 80.0
    assert resultado_contrato(fuera, "C07") == "FALLA"
    return "temperatura de 80 C detectada por C07"


def x07():
    crudo = crudo_sintetico(n=6, estaciones=("station1",))
    crudo.loc[2, "Date"] = "31-02-2022"          # fecha imposible
    resultado = limpieza.PipelineLimpieza(crudo, verboso=False).ejecutar()
    assert int((~resultado["fila_imputada"]).sum()) == 5, "la fila con fecha invalida no se descarto"
    assert int(resultado["fila_imputada"].sum()) == 1, "el instante no quedo como hueco explicito"
    return "31-02-2022 se descarta y su instante queda como hueco explicito"


def x08():
    crudo = crudo_sintetico(n=6, estaciones=("station1",))
    crudo.loc[3, "TEMP"] = "#VALUE!"
    pipe = limpieza.PipelineLimpieza(crudo, verboso=False)
    resultado = pipe.ejecutar()
    assert pd.api.types.is_float_dtype(resultado["temperatura"])
    assert "#VALUE!" in pipe.bitacora().loc[1, "detalle"], "el centinela no quedo registrado"
    return "'#VALUE!' registrado en la bitacora y convertido sin romper el casting"


def x09():
    debe_lanzar(ValueError, lambda: limpieza.escalar_variables(df, metodo="robusto"), contiene="no soportado")
    debe_lanzar(KeyError, lambda: limpieza.escalar_variables(df, columnas=["salinidad"]))
    return "metodo desconocido -> ValueError; columna ausente -> KeyError"


probar("Excepcion", "X01 Archivo de datos inexistente", x01)
probar("Excepcion", "X02 Dataset vacio", x02)
probar("Excepcion", "X03 Columna obligatoria ausente", x03)
probar("Excepcion", "X04 Tipo de dato incorrecto", x04)
probar("Excepcion", "X05 Clave temporal duplicada", x05)
probar("Excepcion", "X06 Valor fuera del dominio fisico", x06)
probar("Excepcion", "X07 Fecha imposible en el archivo crudo", x07)
probar("Excepcion", "X08 Centinela de Excel en el archivo crudo", x08)
probar("Excepcion", "X09 Parametros invalidos en el escalado", x09)

[ OK  ] Excepcion | X01 Archivo de datos inexistente
          -> FileNotFoundError con instrucciones de descarga
[ OK  ] Excepcion | X02 Dataset vacio
          -> C01 FALLA y assert_valido lanza ErrorValidacion
[ OK  ] Excepcion | X03 Columna obligatoria ausente
          -> columna ausente detectada por C02


[ OK  ] Excepcion | X04 Tipo de dato incorrecto
          -> variable como texto detectada por C03
[ OK  ] Excepcion | X05 Clave temporal duplicada
          -> clave (estanque, timestamp) repetida detectada por C05
[ OK  ] Excepcion | X06 Valor fuera del dominio fisico
          -> temperatura de 80 C detectada por C07


[ OK  ] Excepcion | X07 Fecha imposible en el archivo crudo
          -> 31-02-2022 se descarta y su instante queda como hueco explicito
[ OK  ] Excepcion | X08 Centinela de Excel en el archivo crudo
          -> '#VALUE!' registrado en la bitacora y convertido sin romper el casting
[ OK  ] Excepcion | X09 Parametros invalidos en el escalado
          -> metodo desconocido -> ValueError; columna ausente -> KeyError


---
## 7. Registro de pruebas

In [9]:
tabla = pd.DataFrame(registro)
tabla

,categoria,prueba,estado,detalle
0,Normal,N01 Dataset procesado supera todos los contratos,OK,10 contratos OK
1,Normal,N02 Conservacion de registros crudo -> procesado,OK,"74,707 = 74,796 - 38 vacias - 51 sin hora"
2,Normal,N03 Rejilla temporal regular por estanque,OK,paso constante de 20 min en los 3 estanques
3,Normal,N04 Sin valores imposibles y tipos correctos,OK,temperatura minima 14.50 C; variables float64
4,Normal,N05 Nulos residuales solo en huecos largos,OK,todos los nulos residuales estan en huecos > 3...
5,Normal,N06 Escalado z-score correcto por estanque,OK,media 0 y desviacion 1 por estanque
6,Normal,N07 Pipeline completo sobre datos sinteticos v...,OK,dataset sintetico de 3 estanques supera los co...
7,Limite,L01 Hueco de exactamente 1 h se interpola,OK,hueco de 3 lecturas: rellenado y exacto
8,Limite,L02 Hueco de 1 h + 20 min no se interpola,OK,hueco de 4 lecturas: intacto (4 nulos)
9,Limite,L03 Nulos en los extremos no se extrapolan,OK,no extrapola antes de la primera ni despues de...


In [10]:
resumen = tabla.pivot_table(index="categoria", columns="estado", values="prueba", aggfunc="count", fill_value=0)
resumen["total"] = resumen.sum(axis=1)
resumen

estado,OK,total
categoria,,
Excepcion,9,9
Limite,6,6
Normal,7,7


In [11]:
ruta = RAIZ / "docs" / "registro_validacion.csv"
tabla.assign(ejecutado=datetime.now().strftime("%Y-%m-%d %H:%M:%S")).to_csv(ruta, index=False, encoding="utf-8")
print("Registro exportado a:", ruta.relative_to(RAIZ))

no_ok = tabla[tabla["estado"] != "OK"]
assert no_ok.empty, f"{len(no_ok)} prueba(s) no superada(s):\n{no_ok.to_string()}"
print(f"{len(tabla)} pruebas ejecutadas, todas superadas.")

Registro exportado a: docs\registro_validacion.csv
22 pruebas ejecutadas, todas superadas.


---
## 8. Conclusiones

1. **El dataset procesado es válido.** Supera los diez contratos de calidad: esquema
   completo, tipos correctos, clave única, orden temporal, valores dentro del rango
   físico, tres estanques y nulos residuales por debajo del umbral.

2. **El pipeline hace exactamente lo que declara.** El número de registros reales
   coincide con el crudo menos los descartes documentados, la rejilla tiene paso
   constante y todos los nulos restantes pertenecen a huecos largos.

3. **Las reglas funcionan en sus bordes.** Un hueco de una hora se interpola y uno de
   una hora y veinte minutos no; un valor en el límite exacto del rango físico se
   conserva y uno una centésima fuera se anula.

4. **Las entradas inválidas fallan de forma controlada.** Cada defecto produce la
   excepción esperada con un mensaje explicativo, o hace fallar el contrato que le
   corresponde, sin interrumpir el diagnóstico del resto.

El registro completo queda en `docs/registro_validacion.csv`. El dataset
`data/processed/ponds_limpio.csv` queda disponible para la Fase 3.